In [ ]:
!pip uninstall -y langchain langchain-core langchain-community

In [ ]:
!pip install -q \
langchain==0.3.25 \
langchain-core==0.3.59 \
langchain-community==0.3.24 \
langchain-groq \
langchain-google-genai==2.1.4\
faiss-cpu \
pypdf \
requests

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
from langchain_groq import ChatGroq

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain
)

from langchain.chains.combine_documents import (
    create_stuff_documents_chain
)

from langchain_core.runnables.history import (
    RunnableWithMessageHistory
)

from langchain_community.chat_message_histories import (
    ChatMessageHistory
)

In [ ]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant"
)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
loader = PyPDFLoader("SystemDesignInterview.pdf")

documents = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

In [ ]:
print(len(docs))

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Given chat history and latest user question "
        "which might reference previous context, "
        "form a standalone question."
    ),

    MessagesPlaceholder("chat_history"),

    ("human", "{input}")
])

In [ ]:
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer the user's questions using the below context.

Context:
{context}
"""
    ),

    MessagesPlaceholder("chat_history"),

    ("human", "{input}")
])

In [ ]:
question_answer_chain = create_stuff_documents_chain(
    llm,
    qa_prompt
)

In [ ]:
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

In [ ]:
store = {}

In [ ]:
def get_session_history(session_id: str):

    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]

In [ ]:
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [ ]:
response = conversational_rag_chain.invoke(
    {
        "input": "What is this PDF about?"
    },

    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)

print(response["answer"])

In [ ]:
response = conversational_rag_chain.invoke(
    {
        "input": "Explain it in simpler words"
    },

    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)

print(response["answer"])

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    response = conversational_rag_chain.invoke(
        {
            "input": user_input
        },
        config={
            "configurable": {
                "session_id": "user1"
            }
        }
    )

    print("AI:", response["answer"])